# Deforestation Detection & Visualization
**Input format:** two 6-band Sentinel-2 GeoTIFFs of the same area, band order `B2, B3, B4, B8, B11, B12` (before date, after date).
**Model class convention:** 0 = non-forest, 1 = forest. "Deforestation" = forest *before* and non-forest *after*.

In [ ]:
!pip install -q rasterio

## Step 1 - Code (U-Net, change detection, visualization)

In [ ]:
import json, math
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import rasterio
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from rasterio.enums import Resampling
from rasterio.features import shapes
from rasterio.transform import from_origin
from rasterio.vrt import WarpedVRT
from rasterio.warp import transform_geom
from scipy import ndimage as ndi

BAND_ORDER = ("B2", "B3", "B4", "B8", "B11", "B12")
NUM_FEATURES = 9          # 6 bands + NDVI + NBR + NDWI
EPS = 1e-6
NO_CHANGE, STABLE_FOREST, LOSS, GAIN, INVALID = 0, 1, 2, 3, 255


# ============================== U-Net ====================================== #
class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    """Input (B, 9, H, W), H and W divisible by 2**depth. Output: logits (B, 2, H, W)."""

    def __init__(self, in_channels=9, num_classes=2, base=32, depth=4):
        super().__init__()
        chans = [base * 2 ** i for i in range(depth + 1)]
        self.encoders = nn.ModuleList()
        cin = in_channels
        for c in chans[:-1]:
            self.encoders.append(DoubleConv(cin, c))
            cin = c
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(chans[-2], chans[-1])
        self.upconvs, self.decoders = nn.ModuleList(), nn.ModuleList()
        for c in reversed(chans[:-1]):
            self.upconvs.append(nn.ConvTranspose2d(c * 2, c, 2, stride=2))
            self.decoders.append(DoubleConv(c * 2, c))
        self.head = nn.Conv2d(chans[0], num_classes, 1)

    def forward(self, x):
        skips = []
        for enc in self.encoders:
            x = enc(x)
            skips.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        for up, dec, skip in zip(self.upconvs, self.decoders, reversed(skips)):
            x = dec(torch.cat([skip, up(x)], dim=1))
        return self.head(x)


# ======================== Reading + features =============================== #
@dataclass
class Scene:
    bands: np.ndarray   # (6, H, W) float32 reflectance 0-1
    valid: np.ndarray   # (H, W) bool
    crs: object
    transform: object

    @property
    def shape(self):
        return self.bands.shape[1:]


def read_scene(path, reference=None):
    """Read a 6-band GeoTIFF; if `reference` is given, warp onto its grid."""
    with rasterio.open(path) as src:
        if src.count < 6:
            raise ValueError(f"{path}: need 6 bands (B2,B3,B4,B8,B11,B12), found {src.count}")
        idx = list(range(1, 7))
        if reference is None:
            arr, crs, transform = src.read(idx, masked=True), src.crs, src.transform
        else:
            h, w = reference.shape
            same = (src.crs == reference.crs and src.width == w and src.height == h
                    and src.transform.almost_equals(reference.transform))
            if same:
                arr = src.read(idx, masked=True)
            else:
                with WarpedVRT(src, crs=reference.crs, transform=reference.transform,
                               width=w, height=h, resampling=Resampling.bilinear) as vrt:
                    arr = vrt.read(idx, masked=True)
            crs, transform = reference.crs, reference.transform
    mask = np.ma.getmaskarray(arr).any(axis=0)
    data = np.ma.filled(arr.astype(np.float32), 0.0)
    bad = ~np.isfinite(data).all(axis=0)
    data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
    if data.max() > 2.0:               # DN values 0-10000
        data = data / 10000.0
    data = np.clip(data, 0.0, 1.0)
    valid = ~(mask | bad) & (data.sum(axis=0) > 0)
    return Scene(data, valid, crs, transform)


def _nd(a, b):
    return (a - b) / (a + b + EPS)


def compute_features(bands):
    b2, b3, b4, b8, b11, b12 = bands
    extra = np.stack([_nd(b8, b4), _nd(b8, b12), _nd(b3, b8)])   # NDVI, NBR, NDWI
    return np.concatenate([bands, extra]).astype(np.float32)


def simple_cloud_mask(bands, dilate_px=3):
    """Rough heuristic. For real work prefer the Sentinel-2 SCL band or s2cloudless."""
    b2, b3, b4, b8, *_ = bands
    vis = (b2 + b3 + b4) / 3.0
    white = (np.maximum(np.maximum(b2, b3), b4) - np.minimum(np.minimum(b2, b3), b4)) / (vis + EPS)
    cloud = (vis > 0.22) & (white < 0.35) & (_nd(b8, b4) < 0.35)
    if cloud.any() and dilate_px > 0:
        cloud = ndi.binary_dilation(cloud, iterations=dilate_px)
    return cloud


def pixel_area_ha(scene):
    t, crs = scene.transform, scene.crs
    dx, dy = abs(t.a), abs(t.e)
    if crs is not None and crs.is_geographic:
        lat = t.f + t.e * scene.shape[0] / 2.0
        return dx * 111_320.0 * math.cos(math.radians(lat)) * dy * 110_574.0 / 10_000.0
    try:
        f = float(crs.linear_units_factor[1])
    except Exception:
        f = 1.0
    return dx * f * dy * f / 10_000.0


# ============================== Predictors ================================= #
# Every predictor: predict(features (9,H,W)) -> forest probability (H,W) in [0,1]
class NDVIPredictor:
    """Training-free baseline (soft NDVI threshold) for testing the pipeline."""

    def __init__(self, threshold=0.5, softness=0.08):
        self.threshold, self.softness = threshold, softness

    def predict(self, feats):
        return (1 / (1 + np.exp(-(feats[6] - self.threshold) / self.softness))).astype(np.float32)


class RandomForestPredictor:
    """Pixel-wise sklearn RandomForest saved with joblib (features = 9 channels, labels 0/1)."""

    def __init__(self, path, chunk=500_000):
        import joblib
        self.model, self.chunk = joblib.load(path), chunk

    def predict(self, feats):
        c, h, w = feats.shape
        x = feats.reshape(c, -1).T
        pos = list(self.model.classes_).index(1)
        out = np.empty(x.shape[0], np.float32)
        for i in range(0, x.shape[0], self.chunk):
            out[i:i + self.chunk] = self.model.predict_proba(x[i:i + self.chunk])[:, pos]
        return out.reshape(h, w)


class UNetPredictor:
    """Sliding-window inference with overlap and Hann-window blending (no tile seams)."""

    def __init__(self, weights=None, tile=256, overlap=0.5, batch_size=8, device=None, model=None):
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.tile, self.stride, self.batch_size = tile, max(1, int(tile * (1 - overlap))), batch_size
        if model is None:
            ckpt = torch.load(weights, map_location=self.device)
            cfg = {"in_channels": NUM_FEATURES, "num_classes": 2, "base": 32, "depth": 4}
            state = ckpt
            if isinstance(ckpt, dict) and "state_dict" in ckpt:
                state = ckpt["state_dict"]
                cfg.update({k: ckpt[k] for k in cfg if k in ckpt})
            model = UNet(**cfg)
            model.load_state_dict(state)
        self.model = model.to(self.device).eval()
        if tile % 16:
            raise ValueError("tile must be divisible by 16")

    def predict(self, feats):
        c, h, w = feats.shape
        t, s = self.tile, self.stride
        ph = (t - h) if h < t else (-(h - t)) % s
        pw = (t - w) if w < t else (-(w - t)) % s
        padded = np.pad(feats, ((0, 0), (0, ph), (0, pw)), mode="reflect")
        H, W = padded.shape[1:]
        w1 = np.hanning(t + 2)[1:-1].astype(np.float32)
        win = np.outer(w1, w1)
        acc, wsum = np.zeros((H, W), np.float32), np.zeros((H, W), np.float32)
        coords = [(y, x) for y in range(0, H - t + 1, s) for x in range(0, W - t + 1, s)]
        with torch.no_grad():
            for i in range(0, len(coords), self.batch_size):
                chunk = coords[i:i + self.batch_size]
                batch = np.stack([padded[:, y:y + t, x:x + t] for y, x in chunk])
                prob = torch.softmax(self.model(torch.from_numpy(batch).to(self.device)), 1)[:, 1].cpu().numpy()
                for (y, x), p in zip(chunk, prob):
                    acc[y:y + t, x:x + t] += p * win
                    wsum[y:y + t, x:x + t] += win
        return (acc / np.maximum(wsum, 1e-8))[:h, :w]


# ===================== Change detection + post-processing ================== #
def remove_small_patches(mask, min_pixels):
    if min_pixels <= 1 or not mask.any():
        return mask
    labels, _ = ndi.label(mask, structure=np.ones((3, 3)))
    keep = np.bincount(labels.ravel()) >= min_pixels
    keep[0] = False
    return keep[labels]


@dataclass
class ChangeResult:
    change_map: np.ndarray
    loss: np.ndarray
    gain: np.ndarray
    stats: dict


def detect_change(p_before, p_after, valid, px_ha, threshold=0.5, margin=0.1, min_area_ha=0.5):
    """loss = clearly forest before (p >= thr+margin) AND clearly non-forest after (p <= thr-margin).
    The uncertain band is ignored, which reduces false positives."""
    hi, lo = threshold + margin, threshold - margin
    loss = (p_before >= hi) & (p_after <= lo) & valid
    gain = (p_before <= lo) & (p_after >= hi) & valid
    st = np.ones((3, 3), bool)
    loss, gain = ndi.binary_opening(loss, st), ndi.binary_opening(gain, st)
    min_px = max(1, math.ceil(min_area_ha / px_ha))
    loss, gain = remove_small_patches(loss, min_px), remove_small_patches(gain, min_px)

    fb, fa = (p_before >= threshold) & valid, (p_after >= threshold) & valid
    cmap = np.full(valid.shape, NO_CHANGE, np.uint8)
    cmap[fb & fa] = STABLE_FOREST
    cmap[gain], cmap[loss], cmap[~valid] = GAIN, LOSS, INVALID

    fb_ha, loss_ha = float(fb.sum() * px_ha), float(loss.sum() * px_ha)
    stats = {
        "pixel_area_ha": px_ha,
        "valid_area_ha": float(valid.sum() * px_ha),
        "forest_before_ha": fb_ha,
        "forest_after_ha": float(fa.sum() * px_ha),
        "forest_loss_ha": loss_ha,
        "forest_gain_ha": float(gain.sum() * px_ha),
        "loss_pct_of_forest_before": 100 * loss_ha / fb_ha if fb_ha > 0 else 0.0,
        "loss_patches": int(ndi.label(loss, structure=np.ones((3, 3)))[1]),
    }
    return ChangeResult(cmap, loss, gain, stats)


# ================================ Outputs ================================== #
def write_raster(path, array, scene, dtype="uint8", nodata=None):
    with rasterio.open(path, "w", driver="GTiff", height=array.shape[0], width=array.shape[1],
                       count=1, dtype=dtype, crs=scene.crs, transform=scene.transform,
                       nodata=nodata, compress="lzw") as dst:
        dst.write(array.astype(dtype), 1)


def loss_to_geojson(loss, scene, px_ha, path):
    labels, n = ndi.label(loss, structure=np.ones((3, 3)))
    sizes = np.bincount(labels.ravel())
    feats = []
    if n:
        for geom, val in shapes(labels.astype(np.int32), mask=labels > 0,
                                transform=scene.transform, connectivity=8):
            if scene.crs is not None:
                geom = transform_geom(scene.crs, "EPSG:4326", geom)
            feats.append({"type": "Feature", "geometry": geom,
                          "properties": {"id": int(val), "area_ha": round(float(sizes[int(val)] * px_ha), 3)}})
    Path(path).write_text(json.dumps({"type": "FeatureCollection", "features": feats}))
    return len(feats)


# ============================= Visualization =============================== #
def _stretch(*stacks):
    v = np.concatenate([b[[2, 1, 0]].reshape(3, -1)[:, ::50].ravel() for b in stacks])
    v = v[v > 0]
    return tuple(np.percentile(v, (2, 98))) if v.size else (0.0, 1.0)


def _rgb(bands, step, stretch):
    rgb = bands[[2, 1, 0]][:, ::step, ::step].transpose(1, 2, 0)
    lo, hi = stretch
    return np.clip((rgb - lo) / max(hi - lo, 1e-6), 0, 1)


def _overlay(rgb, mask, color, alpha):
    out = rgb.copy()
    for i, c in enumerate(color):
        out[..., i] = np.where(mask, (1 - alpha) * out[..., i] + alpha * c, out[..., i])
    return out


def plot_change(before, after, res, out_png=None, max_dim=2000):
    h, w = before.shape
    step = max(1, max(h, w) // max_dim)
    sl = (slice(None, None, step), slice(None, None, step))
    st = _stretch(before.bands, after.bands)
    rgb_b, rgb_a = _rgb(before.bands, step, st), _rgb(after.bands, step, st)
    ov = _overlay(rgb_a, (res.change_map == STABLE_FOREST)[sl], (0.1, 0.6, 0.2), 0.35)
    ov = _overlay(ov, res.gain[sl], (0.1, 0.3, 0.9), 0.8)
    ov = _overlay(ov, res.loss[sl], (0.9, 0.1, 0.1), 0.85)
    s = res.stats
    fig, axes = plt.subplots(1, 3, figsize=(18, 6.5))
    for ax, img, title in zip(axes, (rgb_b, rgb_a, ov),
                              ("Before (true colour)", "After (true colour)", "Detected forest change")):
        ax.imshow(img, interpolation="nearest")
        ax.set_title(title, fontsize=13)
        ax.axis("off")
    axes[2].legend(handles=[Patch(color=(0.9, 0.1, 0.1), label="Forest loss"),
                            Patch(color=(0.1, 0.3, 0.9), label="Forest gain"),
                            Patch(color=(0.1, 0.6, 0.2), alpha=0.5, label="Stable forest")],
                   loc="lower right", framealpha=0.9)
    fig.suptitle(f"Forest loss: {s['forest_loss_ha']:,.1f} ha ({s['loss_pct_of_forest_before']:.2f}% of forest before)"
                 f" | {s['loss_patches']} patches | forest {s['forest_before_ha']:,.0f} ha -> {s['forest_after_ha']:,.0f} ha",
                 fontsize=13)
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    if out_png:
        fig.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.show()


def run_change_pipeline(before_path, after_path, out_dir, predictor,
                        threshold=0.5, margin=0.1, min_area_ha=0.5, mask_clouds=True):
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    before = read_scene(before_path)
    after = read_scene(after_path, reference=before)      # warped onto the 'before' grid
    px_ha = pixel_area_ha(before)
    valid = before.valid & after.valid
    if mask_clouds:
        valid &= ~simple_cloud_mask(before.bands) & ~simple_cloud_mask(after.bands)
    p_before = predictor.predict(compute_features(before.bands))
    p_after = predictor.predict(compute_features(after.bands))
    res = detect_change(p_before, p_after, valid, px_ha, threshold, margin, min_area_ha)
    write_raster(out / "change_map.tif", res.change_map, before, nodata=INVALID)
    write_raster(out / "forest_prob_before.tif", p_before, before, "float32")
    write_raster(out / "forest_prob_after.tif", p_after, after, "float32")
    res.stats["polygons"] = loss_to_geojson(res.loss, before, px_ha, out / "forest_loss.geojson")
    (out / "stats.json").write_text(json.dumps(res.stats, indent=2))
    plot_change(before, after, res, out / "change_map.png")
    return res.stats


print("Code loaded. Torch device:", "cuda" if torch.cuda.is_available() else "cpu")

## Step 2 - Settings
* `USE_DEMO = True` runs on generated demo data (expected loss = **116 ha**).
* For real data set `USE_DEMO = False`; the notebook will ask you to upload the *before* and *after* GeoTIFFs (or put your Drive paths in `BEFORE_PATH` / `AFTER_PATH`).
* `MODEL`: `"ndvi"` (no training needed, simple baseline), `"unet"` (needs `WEIGHTS_PATH`, a `.pt` file from your teammate), or `"rf"` (needs a `.joblib` file).

In [ ]:
USE_DEMO = True
MODEL = "ndvi"            # "ndvi" | "unet" | "rf"
WEIGHTS_PATH = None       # e.g. "/content/unet.pt" (leave None to be asked for an upload)
BEFORE_PATH = None        # e.g. "/content/drive/MyDrive/before.tif"
AFTER_PATH = None
OUT_DIR = "results"

THRESHOLD = 0.5           # forest probability cut-off
MARGIN = 0.1              # bigger = fewer false positives
MIN_AREA_HA = 0.5         # ignore loss patches smaller than this
MASK_CLOUDS = True

## Step 3 - Get the data

In [ ]:
def make_demo_data():
    """Synthetic 5.12 km x 5.12 km scene (10 m pixels) with two clearings: 80 ha + 36 ha."""
    rng = np.random.default_rng(0)
    H = W = 512
    FOREST = [0.03, 0.05, 0.03, 0.35, 0.15, 0.07]
    CLEARED = [0.08, 0.11, 0.14, 0.22, 0.30, 0.25]
    WATER = [0.04, 0.05, 0.03, 0.02, 0.01, 0.01]

    def scene(boxes):
        img = np.empty((6, H, W), np.float32)
        for i in range(6):
            img[i] = FOREST[i]
        img[:, 400:480, 50:150] = np.array(WATER)[:, None, None]
        for y0, y1, x0, x1 in boxes:
            img[:, y0:y1, x0:x1] = np.array(CLEARED)[:, None, None]
        img *= rng.normal(1.0, 0.04, img.shape).astype(np.float32)
        return (np.clip(img, 0, 1) * 10000).astype(np.uint16)

    for name, boxes in (("demo_before.tif", []), ("demo_after.tif", [(100, 200, 100, 180), (300, 360, 300, 360)])):
        with rasterio.open(name, "w", driver="GTiff", height=H, width=W, count=6, dtype="uint16",
                           crs="EPSG:32648", transform=from_origin(500000, 1300000, 10, 10)) as dst:
            dst.write(scene(boxes))
    return "demo_before.tif", "demo_after.tif"


def upload_files(prompt):
    from google.colab import files          # only exists in Colab
    print(prompt)
    up = files.upload()
    return list(up.keys())


if USE_DEMO:
    BEFORE_PATH, AFTER_PATH = make_demo_data()
    print("Demo data created. Expected forest loss: 116 ha")
else:
    if not (BEFORE_PATH and AFTER_PATH):
        names = upload_files("Select BOTH GeoTIFFs: first the BEFORE image, then the AFTER image "
                             "(they are used in the order shown below).")
        names = sorted(names)
        BEFORE_PATH, AFTER_PATH = names[0], names[1]
        print("Using BEFORE =", BEFORE_PATH, "| AFTER =", AFTER_PATH,
              "\n(files are ordered alphabetically - rename them e.g. 1_before.tif / 2_after.tif if this is wrong)")

if MODEL in ("unet", "rf") and not WEIGHTS_PATH:
    WEIGHTS_PATH = upload_files(f"Upload the {MODEL} weights file (.pt / .joblib)")[0]

## Step 4 - Run detection and visualize

In [ ]:
if MODEL == "unet":
    predictor = UNetPredictor(WEIGHTS_PATH)
elif MODEL == "rf":
    predictor = RandomForestPredictor(WEIGHTS_PATH)
else:
    predictor = NDVIPredictor()

stats = run_change_pipeline(BEFORE_PATH, AFTER_PATH, OUT_DIR, predictor,
                            THRESHOLD, MARGIN, MIN_AREA_HA, MASK_CLOUDS)
print(json.dumps(stats, indent=2))

## Step 5 - Download the results
Files in `results/`: `change_map.png` (figure), `change_map.tif` (0 no change, 1 stable forest, 2 loss, 3 gain, 255 invalid),
`forest_loss.geojson` (loss polygons, open in QGIS / Google Earth), `stats.json`, and the forest-probability maps.

In [ ]:
import shutil
shutil.make_archive("results", "zip", OUT_DIR)
try:
    from google.colab import files
    files.download("results.zip")
except ImportError:
    print("Saved results.zip")